# LLM Fundamentals

**Module:** 05 — LLM Fundamentals

What large language models are, how the field evolved, and the training stack from pre-training through instruction tuning, alignment, and RLHF.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define an LLM as a next-token predictor with emergent capabilities
- Place modern LLMs in the historical evolution of language models
- Outline the training stages: pre-train → instruct → align
- Explain RLHF/preference optimization at a practitioner level
- Know which behaviors come from which training stage


## What is an LLM?

**Definition.** A **Large Language Model** is a neural sequence model (usually a Transformer decoder) trained on vast text to predict the next token, yielding general-purpose language capabilities.

**Why it matters.** LLMs are the core engine behind chat, coding copilots, RAG generators, and agents.

**How it works.** Tokenize text → embed tokens → stack Transformer blocks → project to vocabulary logits → sample/argmax.

**Intuition.** A supercharged autocomplete that absorbed enough of the world to simulate skills.

**Common pitfalls.**
- Equating 'fluent' with 'true'
- Assuming one model fits all tasks/costs
- Ignoring tokenization effects on numbers/code

**When to use.** Any task where flexible language/code generation or understanding is central.

```mermaid
flowchart LR
  T[Text] --> Tok[Tokens]
  Tok --> TR[Transformer stack]
  TR --> L[Logits over vocab]
  L --> S[Decode next token]
  S --> Tok
```


In [ ]:
# Demo 1 — next-token prediction as a probability table
import numpy as np
vocab = ["the", "cat", "sat", "mat", "dog", "."]
# toy logits for continuing: "the cat sat on the"
logits = np.array([0.2, 0.1, 0.1, 2.5, 0.3, 0.2])
probs = np.exp(logits - logits.max()); probs /= probs.sum()
for t, p in sorted(zip(vocab, probs), key=lambda x: -x[1])[:4]:
    print(f"{t:4s} {p:.3f}")


In [ ]:
# Demo 2 — capabilities vs specification
capabilities = {
    "translation": "emergent from multilingual pretraining",
    "coding": "emergent + code-heavy data + instruct",
    "tool use": "mostly post-trained / prompted",
    "your private policy facts": "NOT in weights — use RAG",
}
for k, v in capabilities.items():
    print(f"{k:28s} {v}")


In [ ]:
# Demo 3 — chat API shape (placeholder)
import json
YOUR_API_KEY = "YOUR_API_KEY"
req = {"model": "gpt-4.1-mini", "messages": [{"role": "user", "content": "Explain LLMs simply."}]}
print(json.dumps(req, indent=2))
print("Authorization: Bearer", YOUR_API_KEY[:8] + "...")
print("response.choices[0].message.content → string")


### Try it yourself — What is an LLM?

1. Explain next-token prediction to a friend without jargon.
2. List three tasks that need RAG rather than a bigger LLM.


## Evolution

**Definition.** Language modeling progressed from n-grams → RNNs/LSTMs → Transformers → scaled LLMs → chat/tool-using systems.

**Why it matters.** Knowing the lineage clarifies which ideas are foundational vs hype cycles.

**How it works.** Scale laws + Transformers + internet text + alignment recipes unlocked chat LLMs.

**Intuition.** Each era removed a bottleneck: memory, parallelism, instruction-following, tools.

**Common pitfalls.**
- Skipping classical baselines when they still win on structured tasks

**When to use.** Use history to pick techniques: retrieval, fine-tunes, or prompting.

### Snapshot

| Era | Core idea | Limit |
|-----|-----------|-------|
| n-gram | Counts | Sparsity |
| RNN | Sequential state | Hard to parallelize |
| Transformer | Attention | Compute/memory |
| LLM | Scale + data | Cost, alignment |


In [ ]:
# Demo 1 — timeline table as data
eras = [
    ("n-gram", "2000s", "count statistics"),
    ("RNN/LSTM", "2013-2017", "sequence memory"),
    ("Transformer", "2017+", "parallel attention"),
    ("LLM scale", "2019+", " emergent abilities"),
    ("Chat/tools", "2022+", "alignment + agents"),
]
for name, when, idea in eras:
    print(f"{when:8s} {name:14s} {idea}")


In [ ]:
# Demo 2 — why attention beat recurrence for scale
seq_len = 4096
rnn_steps = seq_len  # sequential
attn_parallel = 1    # matrix ops parallelize on GPU
print({"rnn_sequential_steps": rnn_steps, "transformer_parallel_depth": attn_parallel})


### Try it yourself — Evolution

1. Name the bottleneck each era primarily removed.


## Training Overview

**Definition.** Modern LLMs are typically trained in stages: **pre-training**, **instruction/supervised fine-tuning**, then **alignment** (RLHF/DPO/etc.).

**Why it matters.** Behaviors you see in products map to stages—knowing which knobs exist prevents magical thinking.

**How it works.** Large unlabeled corpus → next-token loss; then curated instruction pairs; then preference optimization.

**Intuition.** First learn language; then learn to be helpful; then learn what humans prefer / policies require.

**Common pitfalls.**
- Fine-tuning when RAG or prompting would suffice
- Aligning without eval harnesses

**When to use.** Plan data/evals for each stage you actually control.

```mermaid
flowchart LR
  P[Pre-training] --> I[Instruction tuning]
  I --> A[Alignment / prefs]
  A --> D[Deploy + eval loop]
```


In [ ]:
# Demo 1 — stage responsibilities
stages = {
  "pretrain": ["fluency", "world priors", "code patterns"],
  "instruct": ["follow formats", "answer questions"],
  "align": ["refuse harm", "match preference", "tone"],
}
for s, items in stages.items():
    print(s, "→", ", ".join(items))


In [ ]:
# Demo 2 — what you can change as an app builder
print("usually: prompts, tools, RAG, sometimes SFT/LoRA")
print("rarely: full pretraining")


In [ ]:
# Demo 3 — loss sketch for next-token
import numpy as np
logits = np.array([0.1, 2.0, 0.3])
target = 1
probs = np.exp(logits - logits.max()); probs /= probs.sum()
nll = -np.log(probs[target] + 1e-12)
print("probs", probs.round(3), "nll", round(float(nll), 4))


### Try it yourself — Training Overview

1. Map a product bug (rude tone vs missing fact) to the likely stage.


## Pre-training

**Definition.** **Pre-training** optimizes next-token prediction on huge corpora to learn general representations.

**Why it matters.** Almost all capabilities and biases originate here; later stages steer rather than replace.

**How it works.** Self-supervised objective on web/books/code; enormous compute; produce a base model.

**Intuition.** Reading the whole library to absorb language statistics and patterns.

**Common pitfalls.**
- Assuming pre-training included your private docs
- Ignoring data cutoffs

**When to use.** You mostly consume pretrained bases; full pre-training is org-scale.


In [ ]:
# Demo 1 — corpus mixture sketch
mixture = {"web": 0.5, "books": 0.15, "code": 0.2, "scientific": 0.1, "other": 0.05}
assert abs(sum(mixture.values()) - 1) < 1e-9
print(mixture)


In [ ]:
# Demo 2 — compute intuition (not real numbers)
params_b, tokens_t = 7, 1.0  # 7B params, 1T tokens
print({"params_B": params_b, "tokens_T": tokens_t, "rough_FLOPs_order": "very large"})


### Try it yourself — Pre-training

1. Why can't pre-training alone guarantee safe corporate assistants?


## Instruction Tuning

**Definition.** **Instruction tuning** (SFT) trains the model on (instruction, response) pairs so it follows user intents.

**Why it matters.** Base models complete text; products need question-answering and formatted obedience.

**How it works.** Curate diverse tasks; supervised fine-tune; evaluate instruction-following and regressions.

**Intuition.** Teaching classroom manners after devouring the library.

**Common pitfalls.**
- Narrow SFT that destroys general skills
- Noisy crowd instructions

**When to use.** Domain formats, tool schemas, or style—when prompts are insufficient.


In [ ]:
# Demo 1 — SFT example record
ex = {"instruction": "Summarize in 2 bullets", "input": "Long memo...", "output": "- Point A\n- Point B"}
print(ex)


In [ ]:
# Demo 2 — format compliance check
def follows_two_bullets(text: str) -> bool:
    lines = [l for l in text.splitlines() if l.strip()]
    return len(lines) == 2 and all(l.strip().startswith("-") for l in lines)
print(follows_two_bullets("- A\n- B"), follows_two_bullets("A\nB"))


In [ ]:
# Demo 3 — chat template sketch
messages = [
  {"role": "system", "content": "You are concise."},
  {"role": "user", "content": "List 2 risks of SFT."},
]
print(messages)


### Try it yourself — Instruction Tuning

1. Draft 3 SFT examples for a JSON-only invoice extractor.


## Alignment

**Definition.** **Alignment** steers models toward human/organizational preferences and policies beyond raw instruction following.

**Why it matters.** Reduces harmful, biased, or disallowed outputs; shapes tone and refusal behavior.

**How it works.** Preference data, constitutions, safety filters, eval red-teams; iterate with metrics.

**Intuition.** Not just 'correct'—also 'appropriate for this product'.

**Common pitfalls.**
- Alignment theater without evals
- Over-refusal hurting usefulness

**When to use.** Any user-facing deployment.


In [ ]:
# Demo 1 — policy categories
policies = ["disallowed_self_harm", "privacy", "weapons", "scam_assist"]
print({p: "refuse_or_safe_complete" for p in policies})


In [ ]:
# Demo 2 — over-refusal vs under-refusal
cases = [("how to build a bomb", "refuse"), ("how to make a cake", "answer")]
for q, expect in cases:
    print(q, "→", expect)


### Try it yourself — Alignment

1. Write two prompts that should refuse and two that should answer for a bank assistant.


## RLHF

**Definition.** **RLHF (Reinforcement Learning from Human Feedback)** fits a reward model on preferences, then optimizes the policy (often PPO) so outputs score higher; newer variants include DPO/ORPO without a separate RL loop.

**Why it matters.** Bridges the gap between 'likely text' and 'text people prefer'.

**How it works.** Collect rankings → train reward model → RL optimize / or direct preference optimization on pairs.

**Intuition.** A taste tester ranks dishes; the chef adjusts recipes toward winners.

**Common pitfalls.**
- Reward hacking
- Tiny preference pools
- Conflicting annotators

**When to use.** Understand vendor models; run only if you have serious preference data/ops.

### Alignment methods (sketch)

| Method | Needs reward model? | Notes |
|--------|---------------------|-------|
| RLHF/PPO | Yes | Classic, complex |
| DPO | No | Preference pairs directly |
| RLAIF | AI feedback | Scales labels |


In [ ]:
# Demo 1 — preference pair
pair = {"prompt": "Explain RAG", "chosen": "clear short answer", "rejected": "rambling wrong answer"}
print(pair)


In [ ]:
# Demo 2 — Bradley-Terry style preference prob
import math
def p_prefer(r_w, r_l):
    return 1 / (1 + math.exp(-(r_w - r_l)))
print(p_prefer(1.2, 0.4), p_prefer(0.2, 1.5))


In [ ]:
# Demo 3 — DPO idea in one line
print("Increase likelihood of chosen, decrease rejected, relative to a reference model.")


### Try it yourself — RLHF

1. Contrast RLHF vs DPO in two bullets each.
2. Name one reward-hacking example in chatbots.


## Glossary

- **base model**: Pretrained model before instruction tuning
- **SFT**: Supervised fine-tuning on instructions
- **RLHF**: RL against a preference-trained reward model


## Summary & Key Takeaways

- LLMs are next-token engines with staged post-training
- Pre-training gives priors; instruct gives obedience; alignment gives policy/taste
- App builders mostly steer with prompts, tools, RAG, and light adaptation
- Preference optimization can be hacked—measure what you care about

### Practice

Write a one-page training-stage map for a product you use daily.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
